# Item-Based Collaborative Filtering

This notebook implements Item-Based Collaborative Filtering on the Netflix Prize dataset.
It covers data filtering, train/test splitting, similarity computation (train-only, no leakage),
rating prediction, RMSE evaluation, MAP@10 evaluation, and recommendation generation.

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.metrics.pairwise import cosine_similarity

## 1. Load Data

In [2]:
df = pd.read_csv("../data/processed/netflix_1000_movies.csv")
df = df[['user_id', 'movie_id', 'rating']]

print("Shape:", df.shape)
df.head()

Shape: (5010199, 3)


,user_id,movie_id,rating
0,1488844,1,3
1,822109,1,5
2,885013,1,4
3,30878,1,4
4,823519,1,3


In [3]:
movies = pd.read_csv(
    "../data/movie_titles.csv",
    header=None,
    encoding="latin1",
    engine="python",
    names=["movie_id", "year", "title"],
    on_bad_lines="skip"
)
print("Movies loaded:", movies.shape)
movies.head()

Movies loaded: (17434, 3)


,movie_id,year,title
0,1,2003.0,Dinosaur Planet
1,2,2004.0,Isle of Man TT 2004 Review
2,3,1997.0,Character
3,4,1994.0,Paula Abdul's Get Up & Dance
4,5,2004.0,The Rise and Fall of ECW


## 2. Filter Data

We keep only users with at least 5 ratings and movies with at least 50 ratings.
This removes noise from very sparse users and movies, which would produce unreliable
similarity scores. The filtered dataset is saved for use in the SVD notebook.

In [4]:
user_counts  = df['user_id'].value_counts()
movie_counts = df['movie_id'].value_counts()

active_users    = user_counts[user_counts  >= 5].index
popular_movies  = movie_counts[movie_counts >= 50].index

filtered_df = df[
    df['user_id'].isin(active_users) &
    df['movie_id'].isin(popular_movies)
].copy()

print("Filtered shape:", filtered_df.shape)
print("Users :", filtered_df.user_id.nunique())
print("Movies:", filtered_df.movie_id.nunique())

Filtered shape: (4660641, 3)
Users : 242848
Movies: 998


In [5]:
filtered_df.to_csv("../data/processed/filtered_netflix.csv", index=False)
print("Saved filtered_netflix.csv")

Saved filtered_netflix.csv


## 3. Train / Test Split

We perform a random 80/20 split at the **rating level**.
The similarity matrix is built exclusively from training data to prevent data leakage.

In [6]:
train_df, test_df = train_test_split(
    filtered_df,
    test_size=0.2,
    random_state=42
)

print("Train:", train_df.shape)
print("Test :", test_df.shape)

Train: (3728512, 3)
Test : (932129, 3)


## 4. Build Item Similarity Matrix (Train Data Only)

Building the full 242k-user pivot matrix would require ~190 GB of RAM.
We sub-sample **20,000 users** from the training set to keep the pivot matrix
manageable. The similarity matrix is computed only from this sub-sample —
**no test data is used at any point here**.

In [7]:
# np.random.seed(42)
# sample_user_ids = np.random.choice(
#     train_df['user_id'].unique(),
#     size=20000,
#     replace=False
# )

# train_small = train_df[train_df['user_id'].isin(sample_user_ids)].copy()

# print("train_small shape:", train_small.shape)
# print("Unique users     :", train_small['user_id'].nunique())
# print("Unique movies    :", train_small['movie_id'].nunique())

In [8]:
# Pivot: rows = users, columns = movies.
# Missing ratings filled with 0 (absence of rating, not a bad rating —
user_movie_matrix = filtered_df.pivot_table(
    index='user_id',
    columns='movie_id',
    values='rating'
).fillna(0)

print(user_movie_matrix.shape)

(242848, 998)


In [9]:
# Cosine similarity between movies  (transpose → movies as rows)
movie_similarity = cosine_similarity(
    user_movie_matrix.T
)

movie_similarity_df = pd.DataFrame(
    movie_similarity,
    index=user_movie_matrix.columns,
    columns=user_movie_matrix.columns
)

print(movie_similarity_df.shape)

(998, 998)


## 5. Rating Prediction

For a given (user, movie) pair we:
1. Look up the user's rated movies **from the training set**.
2. Restrict to movies also present in the similarity matrix.
3. Pick the top-K most similar movies (k=50) as neighbours to avoid diluting
   the signal with very dissimilar items.
4. Compute the similarity-weighted average rating.

In [10]:
# Pre-build a lookup: user_id → DataFrame of (movie_id, rating) from train
# user_ratings_train = {
#     uid: grp[['movie_id', 'rating']].values
#     for uid, grp in train_df.groupby('user_id')
# }

GLOBAL_MEAN = train_df['rating'].mean()
TOP_K_NEIGHBOURS = 50

In [11]:
def predict_rating(user_id, movie_id):

    user_history = train_df[
        train_df['user_id'] == user_id
    ]

    watched_movies = user_history[
        'movie_id'
    ].values

    if len(watched_movies) == 0:
        return GLOBAL_MEAN

    similarities = movie_similarity_df.loc[
        movie_id,
        watched_movies
    ]

    ratings = user_history[
        'rating'
    ].values

    if similarities.sum() == 0:
        return ratings.mean()

    return np.dot(
        similarities,
        ratings
    ) / similarities.sum()

## 6. RMSE Evaluation

We evaluate on a **20,000-row sample** of the test set. The full test set has ~932k
rows — evaluating every row with the Python loop would take hours. 20,000 gives a
stable RMSE estimate while remaining practical.

In [12]:
test_sample = test_df.sample(20000, random_state=42)

predictions = [
    predict_rating(row['user_id'], row['movie_id'])
    for _, row in test_sample.iterrows()
]

rmse = np.sqrt(mean_squared_error(test_sample['rating'], predictions))
print(f"Item-Based CF  RMSE: {rmse:.4f}")

Item-Based CF  RMSE: 1.0017


The RMSE measures how far predicted ratings deviate from actual ratings on held-out data.
An RMSE near 1.0 on a 1–5 scale means predictions are off by roughly one star on average,
which is typical for CF models on sparse datasets.

## 7. MAP@10 Evaluation

MAP@10 (Mean Average Precision at 10) measures ranking quality: are the movies the user
would actually like appearing near the top of the recommendation list?

**Relevance definition:** a movie is relevant if the user's actual rating is **≥ 3.5**.
Since all ratings are integers this is equivalent to rating ≥ 4, matching the competition spec.

We evaluate MAP@10 on a random sample of test users. A movie is considered relevant if the user's actual rating is ≥ 4.

In [13]:
def average_precision_at_k(recommended, relevant, k=10):
    """
    Compute Average Precision@K.

    Parameters
    ----------
    recommended : list of movie_ids in ranked order
    relevant    : set of movie_ids the user actually liked
    k           : cutoff
    """
    score = 0.0
    hits  = 0

    for i, movie in enumerate(recommended[:k], start=1):
        if movie in relevant:
            hits  += 1
            score += hits / i

    if not relevant:
        return 0.0

    return score / min(len(relevant), k)

In [29]:
sample_users = (
    test_df['user_id']
    .drop_duplicates()
    .sample(50, random_state=42)
)

In [21]:
def recommend_for_user(user_id, top_n=10):
    """Return top_n (movie_id, predicted_score) tuples for user_id."""
    watched = set(train_df[train_df['user_id'] == user_id]['movie_id'])

    candidates = set(movie_similarity_df.index) - watched
    scored = [
        (movie, predict_rating(user_id, movie))
        for movie in candidates
    ]
    scored.sort(key=lambda x: x[1], reverse=True)
    return scored[:top_n]

In [22]:
ap_scores = []

for i, user in enumerate(sample_users):

    if i % 5 == 0:
        print(f"Processing user {i}")

    user_test = test_df[
        test_df['user_id'] == user
    ]

    relevant_movies = set(
        user_test[
            user_test['rating'] >= 4
        ]['movie_id']
    )

    if len(relevant_movies) == 0:
        continue

    recs = recommend_for_user(
        user,
        top_n=10
    )

    if len(recs) == 0:
        continue

    recommended_movies = [
        movie
        for movie, _
        in recs
    ]

    ap = average_precision_at_k(
        recommended_movies,
        relevant_movies,
        k=10
    )

    ap_scores.append(ap)

print(
    "Users evaluated:",
    len(ap_scores)
)

if len(ap_scores) > 0:
    map10 = np.mean(ap_scores)

    print(
        f"Item-Based CF MAP@10: {map10:.4f}"
    )

    print(
        f"Max AP: {max(ap_scores):.4f}"
    )
else:
    print("No valid users evaluated.")

Processing user 0
Processing user 5
Processing user 10
Processing user 15
Processing user 20
Processing user 25
Processing user 30
Processing user 35
Processing user 40
Processing user 45
Processing user 50
Processing user 55
Processing user 60
Processing user 65
Processing user 70
Processing user 75
Processing user 80
Processing user 85
Processing user 90
Processing user 95
Users evaluated: 92
Item-Based CF MAP@10: 0.0163
Max AP: 1.0000


MAP@10 scores on sparse datasets with limited interaction history are typically low.
The key comparison is against SVD (see `SVD_Model.ipynb`) and against a random baseline.
Popularity bias is a known weakness of item-CF: the model tends to recommend frequently
co-rated movies, which are often already well-known titles.

## 8. Sample Recommendations

In [23]:
def show_user_history(user_id, n=10):
    """Display the top-n rated movies from a user's training history."""
    history = (
        train_df[train_df['user_id'] == user_id]
        .merge(movies, on='movie_id', how='left')
        [['movie_id', 'title', 'rating']]
        .sort_values('rating', ascending=False)
        .head(n)
    )
    return history


def recommend_for_user_named(user_id, top_n=10):
    """Return a formatted recommendation DataFrame with movie titles."""
    recs = recommend_for_user(user_id, top_n)

    rec_df = (
        pd.DataFrame(recs, columns=['movie_id', 'predicted_rating'])
        .merge(movies, on='movie_id', how='left')
    )

    # Drop recommendations where the title lookup failed
    rec_df = rec_df[rec_df['title'].notna()].reset_index(drop=True)

    return rec_df[['movie_id', 'title', 'predicted_rating']]

In [25]:
sample_user = (
    train_df['user_id']
    .drop_duplicates()
    .sample(1, random_state=42)
    .iloc[0]
)

print("=" * 50)
print(f"USER {sample_user} — Watch History")
print("=" * 50)
display(show_user_history(sample_user))

print()

print("=" * 50)
print(f"USER {sample_user} — Top-10 Recommendations")
print("=" * 50)
display(recommend_for_user_named(sample_user))

USER 562621 — Watch History


,movie_id,title,rating
15,312,High Fidelity,5
7,175,Reservoir Dogs,4
5,443,Rabbit-Proof Fence,4
6,357,House of Sand and Fog,4
9,33,Aqua Teen Hunger Force: Vol. 1,4
10,872,Seven Samurai,4
8,311,Ed Wood,4
4,457,Kill Bill: Vol. 2,4
11,798,Jaws,4
12,329,Dogma,4



USER 562621 — Top-10 Recommendations


,movie_id,title,predicted_rating
0,556,New Waterford Girl,3.803481
1,413,Igby Goes Down,3.798816
2,773,He Died with a Felafel in His Hand,3.794371
3,414,Girl,3.793220
4,369,Playing Mona Lisa,3.793183
5,331,Chasing Amy,3.791163
6,97,Mostly Martha,3.788105
7,516,Monsoon Wedding,3.786369
8,183,IFilm: Deranged,3.786231


### Why these recommendations?

Item-Based CF is **explainable by design**: each recommendation is driven by the similarity
between the candidate movie and the movies the user has already rated highly. For example,
if a user rated *Back to the Future Part III* highly, movies whose rating vectors are most
similar (i.e. frequently co-rated the same way by other users) will rank near the top.

## Limitations

This implementation uses a dense user-item matrix and cosine similarity between movies. While effective on a reduced subset of the Netflix Prize Dataset, the approach becomes computationally expensive as the number of users and items grows.

Additionally, the model may exhibit popularity bias, where highly-rated and frequently-rated movies dominate recommendation lists. Future work could incorporate sparse matrix techniques, neighborhood selection, or matrix factorization methods to improve scalability and recommendation quality.


In [27]:
user_counts = (
    train_df.groupby('user_id')
    .size()
    .sort_values()
)

cold_user = user_counts.index[0]

print(
    "Ratings by user:",
    user_counts.iloc[0]
)

display(
    show_user_history(cold_user)
)

display(
    recommend_for_user_named(cold_user)
)

Ratings by user: 1


,movie_id,title,rating
0,758,Mean Girls,3


,movie_id,title,predicted_rating
0,58,Dragonheart,3.604143
1,65,Lost in the Pershing Point Hotel,3.604143
2,141,Goddess of Mercy,3.604143
3,237,Broken Blossoms,3.604143
4,356,Look at Me,3.604143
5,547,Kuffs,3.604143
6,565,Tom Petty and the Heartbreakers: Live at the O...,3.604143
7,1,Dinosaur Planet,3.000000
8,10,Fighter,3.000000
9,25,Inspector Morse 31: Death Is Now My Neighbour,3.000000
